# 13 - Deployment analysis: benign-population sensitivity and latency

**CPU fine. Run all; idempotent.** Two questions a resolver operator asks that
the main evaluation does not answer, both computable from what already exists:

1. **Does the false-positive rate depend on how popular a benign domain is?**
   Tranco is one view of "benign"; a resolver sees the long tail. Tranco rank
   is joined to the saved trust-score predictions and FPR@95%TPR is reported
   per popularity band. No model is retrained.
2. **What does scoring cost?** Wall-clock per domain for lexical extraction,
   certificate-feature assembly and trust-score inference, measured on this
   runtime. Network costs (TLS probe, DNS/TLSA lookups) are reported from the
   collection ledgers where they were recorded.

Outputs: `table_benign_by_popularity.csv`, `table_latency.csv`.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard xgboost

In [ ]:
import pandas as pd, numpy as np, time, json
from pathlib import Path
from sklearn.metrics import roc_curve
from src.evaluate import predictions
PRED_DIR = Path(P['artifacts']['predictions']); TAB = Path(P['results']['tables']); RAW = Path(P['data']['raw'])
FD = 'family_disjoint_v1'; SEEDS = (42,43,44)

tranco = sorted(RAW.glob('tranco_*.csv'))[-1]
rank = pd.read_csv(tranco, header=None, names=['rank','domain'])
from src.data.universe import registrable
rank['domain'] = rank['domain'].map(registrable)
rank = rank.drop_duplicates('domain').set_index('domain')['rank']
print('tranco', tranco.name, len(rank))

## 1. Benign false-positive rate by popularity band

The 95%-recall threshold is fixed on the whole test set (as deployed); the
false-positive rate is then read off within each band. If the long tail is
blocked more often than the head, the paper says so.

In [ ]:
bands = [(1,1_000,'top 1K'),(1_001,10_000,'1K-10K'),(10_001,100_000,'10K-100K'),(100_001,10**9,'100K+')]
rows = []
for split in [FD, 'random_v1']:
    for sd in SEEDS:
        d = predictions.load(f'trustscore_{split}_s{sd}', PRED_DIR)
        d['rank'] = d['domain'].map(rank)
        y, sc = d['true_label'].values, d['calibrated_score'].values
        fpr, tpr, thr = roc_curve(y, sc); t = thr[np.searchsorted(tpr, 0.95, 'left')]
        ben = d[d.true_label==0]
        for lo, hi, name in bands:
            b = ben[(ben['rank']>=lo)&(ben['rank']<=hi)]
            if len(b) < 50: continue
            rows.append({'split':split,'seed':sd,'band':name,'n_benign':len(b),
                         'fpr_at_95tpr':float((b['calibrated_score']>=t).mean()),
                         'has_cert_rate':float(b['has_certificate'].astype(bool).mean())})
pop = pd.DataFrame(rows).groupby(['split','band'])[['n_benign','fpr_at_95tpr','has_cert_rate']].agg(['mean','std']).round(4)
display(pop); pop.to_csv(TAB/'table_benign_by_popularity.csv')

## 2. Latency

Per-domain wall-clock, median and p95 over 5,000 probe-universe domains, on
this runtime (hardware is recorded alongside). Network stages come from the
ledgers' timestamps, which record attempt and completion times per host.

In [ ]:
import platform, xgboost as xgb
from src.features import lexical, build as fbuild
from src.features import certificate as cert_features
X = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
probe = pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")
sample = X[X['domain'].isin(set(probe['domain']))].sample(5000, random_state=42)

# lexical extraction
lexical.fit_ngram_model(X[X.label==0]['domain'].head(200_000))
t0=time.perf_counter(); _ = lexical.extract_frame(sample['domain'].values); lex_ms = (time.perf_counter()-t0)*1000/len(sample)
lat = [ (time.perf_counter(), lexical.extract(d), time.perf_counter()) for d in sample['domain'].values[:2000] ]
lex_each = np.array([ (c-a)*1000 for a,_,c in lat ])

# certificate-feature assembly (pure transformation of stored observations)
from src.utils.io import read_shards
tls = read_shards(f"{P['data']['collected']}/tls_probe", 'tls_probe').drop_duplicates('domain', keep='last')
t0=time.perf_counter(); _ = cert_features.build(tls); cert_ms = (time.perf_counter()-t0)*1000/len(tls)

# trust-score inference
m = xgb.XGBClassifier(); m.load_model(str(Path(P['artifacts']['models'])/f'fusion_c_fused_{FD}_s42.json'))
feats = m.get_booster().feature_names; cats = [f for f,t in zip(feats, m.get_booster().feature_types) if t=='c']
Xs = sample[[f for f in feats if f not in cats]].apply(pd.to_numeric, errors='coerce').astype(np.float32)
for c in cats: Xs[c] = sample[c].astype('category')
Xs = Xs[feats]
t0=time.perf_counter(); _ = m.predict_proba(Xs); inf_batch_ms = (time.perf_counter()-t0)*1000/len(Xs)
single = []
for i in range(300):
    t0=time.perf_counter(); m.predict_proba(Xs.iloc[[i]]); single.append((time.perf_counter()-t0)*1000)
single = np.array(single)

rows = [
 {'stage':'lexical feature extraction','median_ms':float(np.median(lex_each)),'p95_ms':float(np.percentile(lex_each,95)),'batch_mean_ms':lex_ms},
 {'stage':'certificate feature assembly (from stored cert)','median_ms':cert_ms,'p95_ms':np.nan,'batch_mean_ms':cert_ms},
 {'stage':'trust-score inference (single domain)','median_ms':float(np.median(single)),'p95_ms':float(np.percentile(single,95)),'batch_mean_ms':inf_batch_ms},
]
# network stages from ledgers (attempt -> completion per host)
import sqlite3
for name, dbfile in [('TLS certificate probe (network)', 'certificate_ledger_backup.db')]:
    db = Path(P['artifacts']['logs'])/dbfile
    if db.exists():
        q = pd.read_sql("SELECT attempted_at, completed_at FROM domains WHERE status='success'", sqlite3.connect(db))
        dt = (pd.to_datetime(q.completed_at) - pd.to_datetime(q.attempted_at)).dt.total_seconds()*1000
        dt = dt[(dt>0)&(dt<60000)]
        if len(dt): rows.append({'stage':name,'median_ms':float(dt.median()),'p95_ms':float(dt.quantile(.95)),'batch_mean_ms':np.nan})
lat_tab = pd.DataFrame(rows).round(3)
lat_tab['runtime'] = platform.processor() or platform.machine()
display(lat_tab); lat_tab.to_csv(TAB/'table_latency.csv', index=False)

---
Both tables are transcribed into the manuscript's deployment subsection. DNS
and TLSA lookup latency is added by notebook `02c_dns_tlsa_collection`, which
records per-lookup timing.